# Census Dataset — Data Preprocessing

This notebook uses the methods defined in `preprocessing.py` and the column names and settings defined in `config.py`, instead of rewriting every operation from scratch.

**Before running:** make sure these files are available:
- `preprocessing.py`
- `config.py`
- `census.csv` (update the path in the loading cell if needed)

We first inspect the data, then strip extra spaces and convert the categorical columns to `category`. Duplicates and outliers are not removed, and no encoding or scaling is applied automatically, so the results can be reviewed first.

## 1. Import Libraries and Methods

We import the core libraries, then the ready-made methods from `preprocessing.py`. The notebook only runs the functions; their definitions stay in a separate file.

In [6]:
import pandas as pd
import numpy as np

from preprocessing import (
    load_data,
    clean_data,
    clean_text,
    convert_categorical_columns,
    missing_values_summary,
    duplicate_count,
    remove_duplicates,
    numerical_summary,
    categorical_summary,
    unique_counts,
    split_features_target,
    save_data
)

from config import CATEGORICAL_COLUMNS, NUMERICAL_COLUMNS, TARGET_COLUMN ,PROCESSED_DATA_PATH

## 2. Load Dataset

We use `load_data()` from `preprocessing.py` to read `census.csv` and store the DataFrame in a variable called `data`.

In [7]:
data = load_data()

print("Dataset loaded successfully.")
print("Shape:", data.shape)
display(data.head())

Dataset loaded successfully.
Shape: (45222, 14)


,age,workclass,education_level,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=50K
1,50,Self-emp-not-inc,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,13.0,United-States,<=50K
2,38,Private,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=50K
3,53,Private,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,<=50K
4,28,Private,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,<=50K


## 3. Inspect Dataset Structure

- `shape` shows the number of rows and columns.
- `columns` lists the column names.
- `info()` shows the column data types and the number of non-null values.

In [8]:
print("Rows and columns:", data.shape)
print("\nColumn names:")
print(data.columns.tolist())

Rows and columns: (45222, 14)

Column names:
['age', 'workclass', 'education_level', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


In [9]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 45222 entries, 0 to 45221
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              45222 non-null  int64  
 1   workclass        45222 non-null  str    
 2   education_level  45222 non-null  str    
 3   education-num    45222 non-null  float64
 4   marital-status   45222 non-null  str    
 5   occupation       45222 non-null  str    
 6   relationship     45222 non-null  str    
 7   race             45222 non-null  str    
 8   sex              45222 non-null  str    
 9   capital-gain     45222 non-null  float64
 10  capital-loss     45222 non-null  float64
 11  hours-per-week   45222 non-null  float64
 12  native-country   45222 non-null  str    
 13  income           45222 non-null  str    
dtypes: float64(4), int64(1), str(9)
memory usage: 4.8 MB


## 4. Check Missing Values

We use `missing_values_summary()` to count the missing values in each column. Having no missing values according to pandas does not necessarily mean the data has no invalid entries, so we continue with the remaining checks.

In [10]:
missing_values_summary(data)

age                0
workclass          0
education_level    0
education-num      0
marital-status     0
occupation         0
relationship       0
race               0
sex                0
capital-gain       0
capital-loss       0
hours-per-week     0
native-country     0
income             0
dtype: int64

## 5. Check Duplicate Rows

`duplicate_count()` returns the number of rows that are exact copies of an earlier row. We check the count and a few examples, but we do not drop anything automatically, because identical feature values alone do not prove that a record is invalid.

In [11]:
print("Number of duplicated rows:", duplicate_count(data))
display(data[data.duplicated()].head())

Number of duplicated rows: 5982


,age,workclass,education_level,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
326,33,Private,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,40.0,United-States,>50K
364,27,Private,Bachelors,13.0,Never-married,Craft-repair,Not-in-family,White,Male,0.0,0.0,50.0,United-States,<=50K
518,24,Private,HS-grad,9.0,Never-married,Handlers-cleaners,Unmarried,Black,Female,0.0,0.0,40.0,United-States,<=50K
525,24,Private,HS-grad,9.0,Never-married,Craft-repair,Own-child,White,Male,0.0,0.0,40.0,United-States,<=50K
619,33,Private,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,40.0,United-States,>50K


## 6. Numerical Features Summary

`numerical_summary()` shows descriptive statistics for the numeric columns, such as the mean, median, minimum, maximum, and quartiles. High values are not automatically errors and should not be removed without review.

In [12]:
numerical_summary(data).T

,age,education-num,capital-gain,capital-loss,hours-per-week
count,45222.000000,45222.000000,45222.000000,45222.000000,45222.000000
mean,38.547941,10.118460,1101.430344,88.595418,40.938017
std,13.217870,2.552881,7506.430084,404.956092,12.007508
min,17.000000,1.000000,0.000000,0.000000,1.000000
25%,28.000000,9.000000,0.000000,0.000000,40.000000
50%,37.000000,10.000000,0.000000,0.000000,40.000000
75%,47.000000,13.000000,0.000000,0.000000,45.000000
max,90.000000,16.000000,99999.000000,4356.000000,99.000000


## 7. Categorical Features Summary

`categorical_summary()` summarizes the categorical columns, for example the number of categories and the most frequent one. The columns used are defined in `config.py`.

In [13]:
categorical_summary(data)

,count,unique,top,freq
workclass,45222,7,Private,33307
education_level,45222,16,HS-grad,14783
marital-status,45222,7,Married-civ-spouse,21055
occupation,45222,14,Craft-repair,6020
relationship,45222,6,Husband,18666
race,45222,5,White,38903
sex,45222,2,Male,30527
native-country,45222,41,United-States,41292
income,45222,2,<=50K,34014


## 8. Clean Text and Convert Categorical Columns

`clean_data()` performs two steps:
1. Removes leading and trailing spaces from the text values.
2. Converts the columns listed in `CATEGORICAL_COLUMNS` to the `category` dtype.

The function returns a copy instead of modifying the original DataFrame, so we assign the result back to `data`.

In [14]:
data = clean_data(data)

print("Data cleaned and categorical columns converted.")
display(data.head())

Data cleaned and categorical columns converted.


,age,workclass,education_level,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=50K
1,50,Self-emp-not-inc,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,13.0,United-States,<=50K
2,38,Private,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=50K
3,53,Private,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,<=50K
4,28,Private,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,<=50K


## 9. Verify Data Types

We check the data types after cleaning to confirm that the categorical columns are now `category` and the numeric columns are still numeric.

In [15]:
data.dtypes

age                   int64
workclass          category
education_level    category
education-num       float64
marital-status     category
occupation         category
relationship       category
race               category
sex                category
capital-gain        float64
capital-loss        float64
hours-per-week      float64
native-country     category
income             category
dtype: object

## 10. Count Distinct Categories

`unique_counts()` shows how many distinct categories each categorical column has. For example, if a column contains `Private` many times plus `Government` and `Self-emp`, it has 3 distinct categories, not as many as the number of rows.

In [16]:
unique_counts(data)

native-country     41
education_level    16
occupation         14
workclass           7
marital-status      7
relationship        6
race                5
sex                 2
income              2
dtype: int64

## 11. Inspect Category Frequencies

`value_counts()` shows how many times each category appears. Here we use it to check the distribution of the target column, `income`.

In [17]:
data[TARGET_COLUMN].value_counts()

income
<=50K    34014
>50K     11208
Name: count, dtype: int64

In [18]:
data[TARGET_COLUMN].value_counts(normalize=True).mul(100).round(2)

income
<=50K    75.22
>50K     24.78
Name: proportion, dtype: float64

## 12. Final Validation

After cleaning, we verify the shape, the missing values, the duplicate rows, and the categorical data types. Note that `clean_data()` does not drop duplicate rows, so their count stays the same.

In [19]:
print("Final shape:", data.shape)
print("Missing values:", int(data.isna().sum().sum()))
print("Duplicate rows (not removed):", duplicate_count(data))
print("\nCategorical data types:")
display(data[CATEGORICAL_COLUMNS].dtypes)

Final shape: (45222, 14)
Missing values: 0
Duplicate rows (not removed): 5982

Categorical data types:


workclass          category
education_level    category
marital-status     category
occupation         category
relationship       category
race               category
sex                category
native-country     category
income             category
dtype: object

In [20]:
saved_path = save_data(data)
print("Cleaned data saved to:", saved_path)

Cleaned data saved to: D:\DEPI\finding-donors-ml\data\processed\census_clean.csv


## Summary

In this notebook we used the functions in `preprocessing.py` to inspect the data, check missing values and duplicates, review the summary statistics, strip extra spaces, and convert the categorical columns.

We have not applied encoding or scaling yet, and we have not removed outliers or duplicates automatically. The next step depends on the analysis results and on the type of model.